In [27]:
pip install unidecode

Note: you may need to restart the kernel to use updated packages.


In [28]:
import pandas as pd
import numpy as np
from unidecode import unidecode

In [29]:

# Danh sách các phần tiền tố (không bao gồm phần 'Bundesliga')
file_parts = [
    "base_info",
    "data_details_Aerial",
    "data_details_Assists",
    "data_details_Blocks",
    "data_details_Cards",
    "data_details_Clearances",
    "data_details_Dribbles",
    "data_details_Fouls",
    "data_details_Goals",
    "data_details_Interception",
    "data_details_Key passes",
    "data_details_Offsides",
    "data_details_Passes",
    "data_details_Possession loss",
    "data_details_Saves",
    "data_details_Shots",
    "data_details_Tackles",
    "data_summary"
]

# Tạo danh sách file dành cho EPL
filenames = [f"{part}_SerieA.csv" for part in file_parts]

# Đọc các file CSV
df_list = [pd.read_csv(file) for file in filenames]

# Ghép lại theo chiều ngang
data = pd.concat(df_list, axis=1)

# Hiển thị kết quả
data.head()


,Unnamed: 0,Name,Club,Age,Height,National,Positions,Unnamed: 0,TotAD,WonAD,...,Mins,Goals,Assists,Yel,Red,SpG,PS%,AerialsWon,MotM,Rating
0,0,Juan Musso,Atletico Madrid,30 years old (06-05-1994),191cm,Argentina,Goalkeeper,0,1,1,...,90,-,-,-,-,-,44.8,1,-,7.80
1,1,Daniele Padelli,Udinese,39 years old (25-10-1985),191cm,Italy,Goalkeeper,1,-,-,...,90,-,-,-,-,-,72.2,-,-,7.47
2,2,Rui Patrício,Atalanta,37 years old (15-02-1988),190cm,Portugal,Goalkeeper,2,-,-,...,180,-,-,-,-,-,86.8,-,-,7.38
3,3,Bremer,Juventus,28 years old (18-03-1997),188cm,Brazil,Defender (Centre),3,3.2,2,...,540,-,-,1,-,0.5,92.8,2,1,7.35
4,4,Christos Mandas,Lazio,23 years old (17-09-2001),189cm,Greece,Goalkeeper,4,0.6,0.6,...,450,-,-,-,-,-,68.1,0.6,1,7.34


In [30]:
data= data.drop(columns= ['Unnamed: 0', 'Name'])
data['League'] = 'SerieA'

In [31]:
count_club = data['Club'].value_counts()
count_club

Club
Monza                    34
Venezia                  33
Parma Calcio 1913        32
Empoli                   32
AC Milan                 31
                         ..
Brentford                 1
Leicester                 1
Jagiellonia Bialystok     1
FC Sion                   1
Hoffenheim                1
Name: count, Length: 61, dtype: int64

In [32]:
data = data[data['Club'].isin(count_club[count_club > 20].index)]

In [33]:
player_count = data['Player'].value_counts()
player_count

Player
Marco Sala             2
Nicola Zalewski        2
Gaetano Castrovilli    2
Cristiano Biraghi      2
Mërgim Vojvoda         2
                      ..
Juan Jesus             1
Victor Nelsson         1
Benjamín Domínguez     1
Lautaro Valenti        1
Dele Alli              1
Name: count, Length: 537, dtype: int64

In [34]:
invalid_player = data[data.duplicated(subset='Player', keep=False)]


for p in invalid_player['Player'].unique():
    
    players = data[data['Player'] == p]
    
    # Tìm phần tử có số phút thi đấu Mins lớn nhất
    player_to_keep = players.loc[players['Mins'].idxmax()]
    
    # Xóa các phần tử còn lại
    data = data.drop(players[players.index != player_to_keep.name].index)




In [35]:
data_info = data.sort_values(by = 'Player', ascending= True)
data_value = pd.read_csv('D:\Python_Code\Project_DS108\ds108\Transfermark \SerieA\SerieA_Value.csv')



In [36]:
#data_value['Name'] = data_value['Name'].apply(unidecode)
#data_info['Player'] = data_info['Player'].apply(unidecode)
# Bỏ dấu, chuyển chữ thường và loại dấu gạch nối
data_value['Name'] = data_value['Name'].apply(lambda x: unidecode(x).lower().replace("-", " "))
# Bỏ dấu, chuyển chữ thường và thay dấu gạch nối bằng dấu cách
data_info['Player'] = data_info['Player'].apply(lambda x: unidecode(x).lower().replace("-", " "))



In [37]:
merged_rows = []
name_not_found = []
not_found_indices = []

for i in range(len(data_info)):
    flag = 0
    for j in range(len(data_value)):
        if data_info.iloc[i]['Player'] == data_value.iloc[j]['Name']:
            # Nối 2 hàng lại theo chiều ngang
            merged = pd.concat([
                data_info.iloc[i],
                data_value.iloc[j].drop('Name')  # bỏ cột trùng
            ])
            merged_rows.append(merged)

            flag = 1
            break
    if flag == 0:
        print('Index = ', i, 'Not Found : ', data_info.iloc[i]['Player'])
        name_not_found.append(data_info.iloc[i]['Player'])
        not_found_indices.append(i)




Index =  0 Not Found :  aaron ciammaglichella
Index =  15 Not Found :  alberto manzoni
Index =  66 Not Found :  benjamin dominguez
Index =  70 Not Found :  bob omoregbe
Index =  95 Not Found :  dailon livramento
Index =  102 Not Found :  daniele baselli
Index =  110 Not Found :  david ankeye
Index =  113 Not Found :  davide bartesaghi
Index =  124 Not Found :  diego pugno
Index =  148 Not Found :  federico accornero
Index =  150 Not Found :  federico cassa
Index =  164 Not Found :  francesco camarda
Index =  165 Not Found :  francesco caputo
Index =  204 Not Found :  ionut radu
Index =  207 Not Found :  ismael konate
Index =  214 Not Found :  jacopo bacci
Index =  235 Not Found :  jose palomino
Index =  251 Not Found :  kevin bonifazi
Index =  277 Not Found :  lorenzo anghele
Index =  283 Not Found :  lorenzo tosto
Index =  284 Not Found :  lorenzo venturino
Index =  295 Not Found :  maat caprini
Index =  297 Not Found :  magnus kofod andersen
Index =  303 Not Found :  marc kempf
Index

In [38]:
for idx, name in zip(not_found_indices, name_not_found):
    last_name_1 = name.split(' ')[-1]
    found = False
    for j in range(len(data_value)):
        last_name_2 = data_value.iloc[j]['Name'].split(' ')[-1]
        if last_name_1 == last_name_2:
            merged = pd.concat([
                data_info.iloc[idx],
                data_value.iloc[j].drop('Name')  # bỏ cột trùng
            ])
            merged_rows.append(merged)
            found = True
            break
    if not found:
        print('Still not found:', name)


Still not found: aaron ciammaglichella
Still not found: alberto manzoni
Still not found: bob omoregbe
Still not found: daniele baselli
Still not found: david ankeye
Still not found: davide bartesaghi
Still not found: diego pugno
Still not found: federico accornero
Still not found: federico cassa
Still not found: francesco camarda
Still not found: francesco caputo
Still not found: ismael konate
Still not found: jacopo bacci
Still not found: kevin bonifazi
Still not found: lorenzo anghele
Still not found: lorenzo tosto
Still not found: lorenzo venturino
Still not found: magnus kofod andersen
Still not found: mattia liberali
Still not found: sergiu perciun
Still not found: thomas campaniello
Still not found: tommaso rubino
Still not found: wylan cyprien


In [39]:
for i in range(len(merged_rows)):
    merged_rows[i] = merged_rows[i].to_dict()
data = pd.DataFrame(merged_rows)

data['Player'] = data['Player'].str.title()
data.to_csv('Final_SerieA.csv')